# Detect objects in driving LiDAR

A perception stack on a car gets one LiDAR sweep every 100 ms and has to answer a small, hard question: *what is around me, where, and which way is it pointing?* The answer is a handful of **oriented 3D boxes**, each with a class and a score.

This notebook takes one real KITTI sweep from the sensor to those boxes:

- what a driving sweep contains, and the frame the detector reads it in
- the pillar encoding a voxel detector needs, and why its range and cell size are what they are
- what a detection head actually emits, and how score thresholding and 3D NMS turn 321 408 anchors into 8 boxes
- the boxes drawn over the sweep, on top of the ground truth
- how a whole split is scored, and what the KITTI difficulty tiers mean

The first three sections run anywhere: they use the sweep committed with the docs and a hand-built detection dict, so no checkpoint is involved. The section marked **The real detector** loads a pretrained checkpoint and is guarded by a file check.

If oriented boxes are new, the [Object detection](../models/detection.md) guide is the shorter version of this page.

In [ ]:
# On Colab: !pip install "torch-pointcloud[pyg-lib]"
import numpy as np
import torch
from plyfile import PlyData

import torch_pointcloud as tp
import torch_pointcloud.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| device:", device)

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

from torch_pointcloud.utils.box3d import box_corners

PLAN, SIDE = (88, -90), (26, -84)  # elevation and azimuth: straight down, and from the driver's side
BOX_EDGES = ((0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7))


def show_cloud(pos, color=None, *, ax=None, title=None, size=1.2, view=SIDE, cmap="viridis"):
    """Scatter a sweep. `pos` is (N, 3); `color` is a per-point scalar, a matplotlib color, or None."""
    if ax is None:
        ax = plt.figure(figsize=(7, 4)).add_subplot(projection="3d")
    p = pos.cpu().numpy()
    c = color.cpu().numpy() if torch.is_tensor(color) else color
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, cmap=cmap, depthshade=False, linewidths=0)
    ax.view_init(elev=view[0], azim=view[1])
    span = p.max(0) - p.min(0)
    ax.set_box_aspect(np.maximum(span, 0.02 * span.max()))  # a sheet of pillars is flat in z, which has no aspect
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax


def show_boxes(ax, boxes, colors, linestyle="-"):
    """Draw (K, 7) oriented boxes on a 3D axes as one wireframe per row, heading included."""
    for corners, color in zip(box_corners(boxes).cpu().numpy(), colors):
        segments = [(corners[a], corners[b]) for a, b in BOX_EDGES]
        ax.add_collection3d(Line3DCollection(segments, colors=color, linewidths=1.2, linestyles=linestyle))

## The sweep and its frame

`docs/assets/data/sample_driving.ply` is one frame of the KITTI 3D object benchmark: a Velodyne HDL-64E sweep, kept in the **LiDAR sensor frame**, with the eight annotated objects that frame carries. Each point has an $(x, y, z)$ and one **intensity** channel, the return strength on 0-1.

In [ ]:
import urllib.request
from pathlib import Path

path = Path("../assets/data/sample_driving.ply")  # in a docs checkout
if not path.exists():
    path = Path("sample_driving.ply")
    url = "https://github.com/arthurdjn/pytorch-pointcloud/raw/main/docs/assets/data/sample_driving.ply"
    if not path.exists():
        urllib.request.urlretrieve(url, path)

ply = PlyData.read(path)
vertex, annotation = ply["vertex"], ply["box"]
sweep = {
    "pos": torch.from_numpy(np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1).astype(np.float32)),
    "intensity": torch.from_numpy(np.asarray(vertex["intensity"]).astype(np.float32)).reshape(-1, 1),
    "box": torch.from_numpy(
        np.stack([annotation[f] for f in ("x", "y", "z", "dx", "dy", "dz", "heading")], axis=1).astype(np.float32)
    ),
    "label": torch.from_numpy(np.asarray(annotation["label"]).astype(np.int64)),
}

pos = sweep["pos"]
print("points:", len(pos))
for axis, name in enumerate(("x forward", "y left  ", "z up    ")):
    print(f"  {name}: {float(pos[:, axis].min()):6.1f} to {float(pos[:, axis].max()):5.1f} m")
horizontal = pos[:, :2].norm(dim=1)
print("max horizontal range:", round(float(horizontal.max()), 1), "m")
print("points within 20 m:", int((horizontal < 20).sum()), "| beyond 40 m:", int((horizontal >= 40).sum()))
print("intensity:", round(float(sweep["intensity"].min()), 2), "to", round(float(sweep["intensity"].max()), 2),
      "| mean", round(float(sweep["intensity"].mean()), 2))
print("ground plane (1st percentile of z):", round(float(pos[:, 2].quantile(0.01)), 2), "m")

16 835 points, out to 80 m, with the sensor at the origin. Three conventions are doing work here and every driving detector in the library assumes them:

- **$+x$ is forward, $+y$ is left, $+z$ is up**, with the origin at the LiDAR. Nothing is centered or normalized: a detector defines its voxel grid in meters in this frame, so recentering the cloud would move the whole scene out of range.
- **The ground is a plane at $z \approx -1.75$ m**, the sensor's mount height. That is why the anchor boxes below sit at fixed heights instead of being regressed from scratch.
- **The sweep is cropped to the front camera's field of view.** KITTI only annotates what the left color camera sees, so the standard protocol (`KITTI(..., fov=True)`) drops the rest of the 360-degree sweep. That is the wedge you see from above.

A box is 7 numbers, $(c_x, c_y, c_z, d_x, d_y, d_z, \theta)$: center, full extents, and a heading counter-clockwise about $+z$ from $+x$. The center is the **box center**, not the ground contact point.

In [ ]:
intensity = sweep["intensity"].squeeze(1)

fig = plt.figure(figsize=(12, 4))
show_cloud(pos, intensity, ax=fig.add_subplot(121, projection="3d"),
           title=f"one KITTI sweep: {len(pos):,} points by return intensity")
show_cloud(pos, intensity, ax=fig.add_subplot(122, projection="3d"), view=PLAN,
           title="the same sweep from above");

![One KITTI sweep drawn twice by return intensity, from the driver's side and from straight above](../assets/tutorials/driving_sweep.png)

The sweep is a stack of laser rings on the road surface and it thins with range exactly the way the counts say: 12 067 of the 16 835 points fall within 20 m and 862 beyond 40 m, so 72% of the data describes the nearest quarter of the range.

Intensity averages 0.27 inside 40 m and 0.05 beyond it. Past that distance a return is barely a return, which is the same fact the point counts report seen a second way.

The right panel is the same sweep from straight above, and that view is where the crop shows itself: the sweep is a wedge, not a disc, because KITTI only annotates what the left color camera sees.

In [ ]:
from torch_pointcloud.datasets.kitti import KITTI_CLASSES

for box, label in zip(sweep["box"], sweep["label"]):
    distance = float(box[:2].norm())
    print(f"  {KITTI_CLASSES[int(label)]:8s} at {distance:5.1f} m"
          f"  size ({float(box[3]):.2f}, {float(box[4]):.2f}, {float(box[5]):.2f}) m"
          f"  heading {float(box[6]):+.2f} rad"
          f"  bottom z {float(box[2] - box[5] / 2):+.2f} m")

Five cars and three cyclists, from 5.4 m to 45.3 m out. Two things to notice. Every box bottom lands within 40 cm of the $-1.75$ m ground plane, which is the regularity anchor-based detectors exploit. And every heading is within 0.1 rad of $0$ or $\pi$: this is a straight road and everything on it is aligned with the lane, which is *not* true of a parking lot and is one reason KITTI numbers travel poorly.

## What a pillar encoder needs

PointPillars does not consume points. It quantizes the sweep onto a 2D grid of vertical **pillars**, stacks the points that fall in each one, and runs a 2D CNN over the resulting bird's-eye feature map. Three numbers define that grid, and the checkpoint fixes all three:

| parameter | value | why |
| --- | --- | --- |
| `point_cloud_range` | $(0, -39.68, -3) \to (69.12, 39.68, 1)$ m | forward half only (KITTI annotates the camera view), 4 m of height, sized so the grid divides evenly |
| `voxel_size` | $(0.16, 0.16, 4.0)$ m | 16 cm in the ground plane; the full height in one cell, which is what makes a *pillar* rather than a voxel |
| `max_num_points` | 32 | points per pillar the encoder reads; the stack is padded below it and truncated above |

`HardVoxelize` does the quantization, and `Cat` builds the per-point feature the encoder appends to $(x, y, z)$. For this checkpoint that feature is the single intensity channel.

In [ ]:
transform = T.Compose([
    T.Cat(keys=["intensity"], dst_key="x", dim=1),
    T.HardVoxelize(
        pos_key="pos",
        feat_key="x",
        voxel_size=(0.16, 0.16, 4.0),
        point_cloud_range=(0.0, -39.68, -3.0, 69.12, 39.68, 1.0),
        max_num_points=32,
        max_num_voxels=40000,
    ),
])

sample = transform({"pos": pos.clone(), "intensity": sweep["intensity"].clone()})
print({k: tuple(v.shape) for k, v in sample.items() if torch.is_tensor(v)})

cells = round(69.12 / 0.16) * round(79.36 / 0.16)
counts = sample["voxel_num_points"]
print("occupied pillars:", len(counts), f"of {cells} grid cells ({100 * len(counts) / cells:.2f}%)")
print("points kept:", int(counts.sum()), "of", len(pos))
print("points per pillar: mean", round(float(counts.float().mean()), 1), "| full (32):", int((counts == 32).sum()))

`voxel` is $(4055, 32, 4)$: 4 055 occupied pillars, up to 32 points each, four channels per point $(x, y, z, \text{intensity})$. `pos_voxel` is the integer grid index $(z, y, x)$ of each pillar and `voxel_num_points` says how many of the 32 slots are real.

The numbers are worth sitting with. The grid has $432 \times 496 = 214\,272$ cells and **1.89% of them are occupied**: a bird's-eye grid over a LiDAR sweep is almost entirely empty, which is exactly why the pillar encoder only ever touches the occupied ones. And 1 114 of the 16 835 points do not survive: 583 fall outside the range (236 beyond 69.12 m, 477 above $z = 1$ m, 130 of them both), and the remaining 531 are the overflow of the 44 pillars that hit the 32-point cap. Both losses are deliberate, and both are invisible unless you count.

In [ ]:
voxel_size = torch.tensor([0.16, 0.16, 4.0])
origin = torch.tensor([0.0, -39.68, -3.0])
centers = sample["pos_voxel"].flip(1).float() * voxel_size + origin + voxel_size / 2

show_cloud(centers, size=0.6, view=PLAN, title=f"{len(centers):,} occupied pillars of 0.16 m");

`pos_voxel` is stored as $(z, y, x)$, so it is flipped before it is scaled back into meters. Every pillar comes out at $z = -1$ m, the middle of the 4 m range, because the grid has exactly one cell in $z$: that is what makes a *pillar* rather than a voxel, and it is why the plot above is a sheet rather than a volume.

Seen from above the 16 cm lattice resolves in the near field: the laser rings of the sweep become rows of cells, and the gaps between rings become empty ones.

## Decoding, with no weights at all

A detection head does not emit boxes. It emits a dense field of proposals: one prediction per anchor, per grid cell, with no filtering whatsoever. `model.decode(out)` unpacks that field into a flat `Detection3D`, and turning it into a usable answer is two more steps that live in *your* code, not the model's, because they belong to the evaluation protocol.

`Detection3D` is a plain `TypedDict` of packed tensors, [PyG](https://pytorch-geometric.readthedocs.io/)-style, so it can be built by hand. This is the whole contract:

| key | shape | meaning |
| --- | --- | --- |
| `boxes` | $(K, 7)$ | $(c_x, c_y, c_z, d_x, d_y, d_z, \theta)$, full extents |
| `labels` | $(K,)$ | class index per box |
| `scores` | $(K,)$ | confidence per box |
| `batch` | $(K,)$ | which scene of the batch each box belongs to |

`Boxes3D` is the same thing without `scores`: that is the shape ground truth takes.

To see the filtering on its own, here is a synthetic head output. Take the eight true boxes, make 40 jittered copies of each with decaying scores, and the result has the shape of what a real head emits: many overlapping proposals per object.

In [ ]:
from torch_pointcloud.utils.box3d import nms3d

generator = torch.Generator().manual_seed(0)
copies = 40
seeds = sweep["box"].repeat_interleave(copies, dim=0)
spread = torch.cat([seeds[:, 3:6] * 0.12, seeds[:, 3:6] * 0.06, torch.full((len(seeds), 1), 0.05)], dim=1)

proposals = {
    "boxes": seeds + torch.randn(len(seeds), 7, generator=generator) * spread,
    "scores": (torch.rand(len(seeds), generator=generator) * 0.5
               + torch.linspace(0.5, 0.0, copies).repeat(len(sweep["box"]))).clamp(0, 1),
    "labels": torch.zeros(len(seeds), dtype=torch.long),
    "batch": torch.zeros(len(seeds), dtype=torch.long),
}
print("proposals:", len(proposals["boxes"]))

kept = proposals["scores"] > 0.5
print("above score 0.50:", int(kept.sum()))

index = nms3d(proposals["boxes"][kept], proposals["scores"][kept], 0.01,
              batch=proposals["batch"][kept], labels=proposals["labels"][kept], rotated=True)
print("after 3D NMS:", len(index))

320 proposals, 156 of them above the score threshold, 8 after non-maximum suppression: one per object.

`nms3d` keeps the highest-scoring box of each overlapping cluster. Three arguments matter:

- **`rotated=True`** suppresses on the exact rotated bird's-eye IoU. The default is the cheaper axis-aligned surrogate, which over-suppresses angled neighbors: two cars parked nose to tail at 45 degrees have overlapping *axis-aligned* boxes and disjoint real ones. Outdoors, always pass it.
- **`labels=`** restricts suppression to boxes of the same class, so a cyclist next to a car does not delete it.
- **`batch=`** runs NMS independently per scene and returns one index tensor over the concatenated input, so a batch of frames never suppresses across frames.

The IoU threshold of `0.01` is the KITTI convention and is unusually strict: with rotated IoU, real objects on a road essentially never overlap, so anything that does is a duplicate.

## The real detector

The rest of this notebook runs `pointpillars.kitti.openpcdet`, a PointPillars trained on the KITTI 3-class split. It needs the pretrained weights in the local model cache; everything above does not.

In [ ]:
from torch_pointcloud.config import MODELS_DIR

weights = Path(MODELS_DIR) / "pointpillars" / "pointpillars.kitti.openpcdet.safetensors"
print("weights present:", weights.exists(), "|", weights)

In [ ]:
model, info = tp.create_model(
    "pointpillars.kitti.openpcdet",
    task="detection",
    pretrained=True,
    return_info=True,
)
model = model.to(device).eval()

print("classes:", info["weights"]["classes"])
print(info["transform"])

`return_info=True` hands back the exact preprocessing the checkpoint was trained with, which is the `Cat` + `HardVoxelize` pair built by hand above. Use `info["transform"]` rather than re-deriving it: a voxel detector is silently wrong if its grid does not match the one it was trained on.

`collate` packs a list of samples into a batch. Pillars are ragged across scenes, so `pos_voxel` is concatenated rather than stacked and the loader synthesizes a `batch_pos_voxel` index naming the scene each pillar came from.

In [ ]:
from torch_pointcloud.utils.data import collate

batch = collate(
    [info["transform"]({"pos": pos.clone(), "intensity": sweep["intensity"].clone()})],
    cat_keys=["pos_voxel"],
)
with torch.no_grad():
    out = model(
        batch["voxel"].to(device),
        batch["pos_voxel"].to(device),
        batch["voxel_num_points"].to(device),
        batch["batch_pos_voxel"].to(device),
    )

print("head output:", {k: tuple(v.shape) for k, v in out.items()})
print("anchors:", tuple(model.head.anchors.shape))

That is the raw head, before anything is decoded, and it is worth reading key by key.

`cls`, `box` and `dir_cls` are the three $1 \times 1$ convolutions of the anchor head, still shaped as a $248 \times 216$ bird's-eye feature map. There are 6 anchors per cell (3 classes, each at $0$ and $\pi/2$), so the channel counts are $6 \times 3 = 18$ class logits, $6 \times 7 = 42$ box residuals and $6 \times 2 = 12$ direction-bin logits.

`batch_cls` and `batch_box` are those same predictions flattened and *already decoded against the anchors*: $248 \times 216 \times 6 = 321\,408$ absolute boxes in meters, and one class logit vector per box. The box residuals encode a center offset normalized by the anchor's base diagonal and log-ratios of the extents, so the head only ever learns a correction to a template of roughly the right size in roughly the right place. The direction bin resolves the $\pi$ ambiguity a regressed heading cannot.

Nothing has been filtered. `decode` flattens this into `Detection3D`.

In [ ]:
detections = model.decode(out)
print({k: tuple(v.shape) for k, v in detections.items()})
print("score range:", round(float(detections["scores"].min()), 4), "to", round(float(detections["scores"].max()), 4))

above = detections["scores"] > 0.1
boxes, scores = detections["boxes"][above], detections["scores"][above]
labels, index = detections["labels"][above], detections["batch"][above]
print("above score 0.10:", int(above.sum()))

index = nms3d(boxes, scores, 0.01, batch=index, rotated=True)
boxes, scores, labels = boxes[index].cpu(), scores[index].cpu(), labels[index].cpu()
print("after 3D NMS:", len(boxes))

final = scores > 0.5
print("above score 0.50:", int(final.sum()))

In [ ]:
CLASS_COLOR = ["tab:orange", "tab:blue", "tab:green"]  # Car, Pedestrian, Cyclist

classes = info["weights"]["classes"]
near = pos[pos[:, 0] < 50]  # the far quarter of the sweep holds no annotated object
scored = detections["boxes"][above].cpu()
scored_labels = detections["labels"][above].cpu()

stages = [(f"{len(scored)} boxes above score 0.10", scored, scored_labels),
          (f"{len(boxes)} boxes after 3D NMS", boxes, labels)]

fig = plt.figure(figsize=(12, 5))
for index, (title, stage_boxes, stage_labels) in enumerate(stages):
    ax = show_cloud(near, "0.6", ax=fig.add_subplot(1, 2, index + 1, projection="3d"),
                    title=title, size=0.5, view=PLAN)
    show_boxes(ax, stage_boxes, [CLASS_COLOR[int(label)] for label in stage_labels])

![Two panels of the same sweep from above, 218 proposals above the score threshold and then the 17 that survive 3D NMS](../assets/tutorials/driving_decode.png)

The collapse is the whole story of decoding: **321 408 anchors, 218 above the score threshold, 17 after NMS, 8 above 0.50**. The left panel shows how tightly the surviving proposals cluster on the real objects, five and ten deep on every car, with a thin scatter of low-scoring boxes elsewhere. The right panel is what NMS leaves, including several low-scoring boxes that only the display threshold removes.

Two thresholds, doing different jobs. The `0.1` before NMS is the *benchmark's* threshold: average precision integrates a precision-recall curve, so it wants the low-scoring tail kept. The `0.5` afterwards is a *display* threshold, for a figure or a downstream planner that needs a decision. Never confuse the two: thresholding at 0.5 before scoring throws away recall the metric would have credited.

The NMS call above leaves `labels=` off, the way `examples/pointpillars_benchmark_kitti.py` does, because the KITTI protocol suppresses across classes. At IoU 0.01 that costs nothing here: the same 8 boxes come out either way.

In [ ]:
for box, score, label in zip(boxes[final], scores[final], labels[final]):
    print(f"  {classes[int(label)]:8s} {float(score):.2f}"
          f"  center ({float(box[0]):5.1f}, {float(box[1]):5.1f}, {float(box[2]):+.2f})"
          f"  size ({float(box[3]):.2f}, {float(box[4]):.2f}, {float(box[5]):.2f})"
          f"  heading {float(box[6]):+.2f}")

Eight boxes: five cars and three cyclists, scored 0.74 to 0.94. The headings come back wrapped into $[0, 2\pi)$ while the KITTI annotations use $(-\pi, \pi]$, so a predicted $+6.23$ and an annotated $-0.05$ are the same direction. Compare on $\cos$ and $\sin$, or wrap one to the other's convention, before calling a heading wrong.

In [ ]:
to_detection = {KITTI_CLASSES.index(name): index for index, name in enumerate(classes)}
truth_labels = torch.tensor([to_detection[int(label)] for label in sweep["label"]])

fig = plt.figure(figsize=(12, 5))
shown = f"{int(final.sum())} predicted boxes over the {len(sweep['box'])} annotations"
for index, (view, angle) in enumerate(((PLAN, "from above"), (SIDE, "from the driver's side"))):
    ax = show_cloud(near, "0.6", ax=fig.add_subplot(1, 2, index + 1, projection="3d"), size=0.5, view=view,
                    title=f"{shown}, {angle}")
    show_boxes(ax, sweep["box"], ["0.35"] * len(sweep["box"]), linestyle="--")
    show_boxes(ax, boxes[final], [CLASS_COLOR[int(label)] for label in labels[final]])

![Eight predicted boxes over the eight annotated boxes on the same sweep, seen from straight above](../assets/tutorials/driving_predictions_bev.png)

Eight predictions over eight annotations, on the same eight objects with the same classes. The predictions are drawn solid and class-colored, the annotations dashed and gray. From above, every box lies along the lane: that is what the direction-bin logits resolved.

The right panel closes in on the nearest 20 m, and there the agreement stops being a number. The predicted box on the nearest car is 34 cm longer than the annotation, its center 30 cm nearer the sensor and 8 cm lower.

![The same eight predicted and annotated boxes seen from the driver's side rather than from above](../assets/tutorials/driving_predictions_perspective.png)

From the side the boxes stand on the road plane and their heights compare directly. Across all eight objects the median center error in the ground plane is 0.10 m and the worst is 0.64 m, on the car at 43.7 m. A LiDAR only ever sees the two faces turned toward it, so the far side of every box is inferred and the extents drift with range.

In [ ]:
from torch_pointcloud.utils.box3d import boxes_iou3d, count_points_in_boxes

overlap = boxes_iou3d(boxes[final], sweep["box"])
returns = count_points_in_boxes(pos, sweep["box"])
for row, (score, label) in enumerate(zip(scores[final], labels[final])):
    column = int(overlap[row].argmax())
    print(f"  {classes[int(label)]:8s} {float(score):.2f}"
          f"  ->  {KITTI_CLASSES[int(sweep['label'][column])]:8s}"
          f"  at {float(sweep['box'][column, :2].norm()):5.1f} m"
          f"  3D IoU {float(overlap[row, column]):.2f}"
          f"  on {int(returns[column]):4d} points")
print("ground-truth boxes recovered at IoU 0.5:", int((overlap.max(dim=0).values > 0.5).sum()), "of", len(sweep["box"]))

All eight objects are found, with 3D IoU from 0.54 to 0.84. The last column counts the LiDAR returns inside each annotation, and the spread is enormous: 1 692 on the car at 9.6 m, **8** on the car at 43.7 m, which is also the weakest match at 0.54. It is not a clean law (the car at 45.3 m matches at 0.84 on 41 returns), but the direction is the one to expect. Out there a box is mostly the anchor template showing through, because the points that would correct it were never returned.

## Scoring a split

One frame proves the pipeline runs; it says nothing about the model. The metric for that is **average precision**, integrated over the precision-recall curve of a whole split, with a per-class IoU threshold. KITTI uses Car at 0.7 and Pedestrian / Cyclist at 0.5, because a cyclist is small enough that 0.7 would be dominated by annotation noise.

In [ ]:
from torch_pointcloud.utils.metrics import average_precision3d

prediction = {
    "boxes": boxes[final],
    "scores": scores[final],
    "labels": labels[final],
    "batch": torch.zeros(int(final.sum()), dtype=torch.long),
}
target = {
    "boxes": sweep["box"],
    "labels": truth_labels,
    "batch": torch.zeros(len(sweep["box"]), dtype=torch.long),
}

metrics = average_precision3d(
    [prediction], [target],
    iou_per_class={0: 0.7, 2: 0.5},
    class_names=classes,
    interpolation="all",
)
print({name: round(value, 3) for name, value in metrics.items()})

`AP/Car` 0.76, `AP/Cyclist` 1.00 on this frame: all three cyclists clear 0.5, and four of the five cars clear 0.7 while the one at 43.7 m lands at 0.54.

Note `interpolation="all"`, which integrates the full curve. The KITTI protocol uses `"r11"` or `"r40"` instead, sampling precision on an 11- or 41-point recall grid. Those grids need thousands of boxes to populate: on a single frame with five cars, recall moves in steps of 0.2 and most grid slots stay empty, so `"r11"` reports a number that looks like a failure and is really an artifact of the sample size. Use `"all"` when you are inspecting one scene, `"r11"` when you are reproducing published KITTI numbers.

**Difficulty tiers.** KITTI splits every annotation into easy / moderate / hard from three per-object fields, all carried in the committed sample: the height of the object's 2D box in the camera image, its occlusion level (0-3) and its truncation fraction. Published KITTI numbers are almost always the *moderate* tier.

In [ ]:
height = torch.from_numpy(np.asarray(annotation["bbox_height"]).astype(np.float32))
occlusion = torch.from_numpy(np.asarray(annotation["occlusion"]).astype(np.int64))
truncation = torch.from_numpy(np.asarray(annotation["truncation"]).astype(np.float32))

easy = (height >= 40) & (occlusion == 0) & (truncation <= 0.15)
moderate = (height >= 25) & (occlusion <= 1) & (truncation <= 0.30) & ~easy
print("easy:", int(easy.sum()), "| moderate:", int(moderate.sum()), "| hard:", int((~easy & ~moderate).sum()))

for name, tier in (("easy", easy), ("moderate", moderate), ("hard", ~easy & ~moderate)):
    for row in tier.nonzero().flatten().tolist():
        print(f"  {name:8s} {KITTI_CLASSES[int(sweep['label'][row])]:8s}"
              f"  2D height {float(height[row]):3.0f} px  occlusion {int(occlusion[row])}"
              f"  truncation {float(truncation[row]):.2f}")

Three easy, four moderate, one hard. The hard one is the car at 5.4 m: the closest object in the frame and the biggest in the image at 201 px, hard only because 75% of it falls outside the image. Difficulty is a property of *visibility*, not of distance.

**Scoring the whole split** is what `examples/pointpillars_benchmark_kitti.py` does. It runs the val split through the same decode, threshold and NMS used above, plus the two rules a single frame does not need: `RelabelBoxes` turns Van and Person_sitting into ignore regions rather than false positives, and predictions whose projected 2D height falls below 25 px are excluded, which is the moderate tier's own rule. Run it with:

```bash
uv run --no-sync python examples/pointpillars_benchmark_kitti.py \
    --root "/path/to/parent" --split-file /path/to/ImageSets/val.txt
```

| source | Car | Pedestrian | Cyclist | mAP |
| --- | --- | --- | --- | --- |
| OpenPCDet model zoo (val, moderate, R11) | 77.28 | 52.29 | 62.68 | 64.08 |
| `torch-pointcloud` | 77.33 | 50.45 | 60.13 | 62.64 |

`examples/second_benchmark_kitti.py` and `examples/pointrcnn_benchmark_kitti.py` are the same script for the other two KITTI checkpoints, and `examples/lion_benchmark_nuscenes.py` is the nuScenes equivalent. Do not rewrite them: they carry the protocol details that make the numbers comparable.

## Choosing a detector

`list_models(task="detection", pretrained=True)` is the full set. For driving LiDAR the shipped checkpoints are:

| checkpoint | dataset | classes | needs |
| --- | --- | --- | --- |
| `pointpillars.kitti.openpcdet` | KITTI | Car, Pedestrian, Cyclist | nothing extra |
| `second.kitti.openpcdet` | KITTI | Car, Pedestrian, Cyclist | `spconv` |
| `pointrcnn.kitti.openpcdet` | KITTI | Car, Pedestrian, Cyclist | nothing extra |
| `pointpillars-multihead.nuscenes.openpcdet` | nuScenes | 10 | nothing extra |
| `second-multihead.nuscenes.openpcdet` | nuScenes | 10 | `spconv` |
| `voxelnext.nuscenes.openpcdet` | nuScenes | 10 | `spconv` |
| `lion-mamba.nuscenes.zhe-liu` | nuScenes | 10 | `spconv`, `mamba` |

They all share the interface used above: `forward` returns raw head output, `decode` returns `Detection3D`, and thresholding plus `nms3d` is yours. What changes is the input contract. The nuScenes checkpoints aggregate 10 sweeps and take a **timestamp** channel alongside intensity, so their transform is `Cat(keys=["intensity", "timestamp"])`; a nuScenes model fed a single sweep with no timestamp will run and predict badly. `info["transform"]` is always the authority.

In [ ]:
print(tp.list_models(task="detection", pretrained=True))

## Next steps

- [Segment a survey-scale LiDAR tile](07-large-scale-lidar.md): the other outdoor problem, where the answer is a label per point and the scene does not fit in one pass.
- [Understand an indoor scene](09-indoor-scene.md): detection and segmentation reconciled into an object inventory, on an indoor room.
- [Object detection](../models/detection.md): the short reference for the box format, `decode` and the box utilities.
- [Detection datasets](../datasets/detection.md): what KITTI, nuScenes, SUN RGB-D and ScanNet each give you.